# JOILang PromptOps: Local Model Benchmark & Render Trace

이 노트북은 JOILang-Server 저장소의 `run_benchmark.py`를 활용하여 실제 로컬 모델 환경에서 blocks 기반 프롬프트 렌더링이 어떻게 동작하는지 한 단계씩 점검하기 위한 디버깅 및 트레이싱 도구입니다.

주요 목적:
1. 로컬 환경 변수 설정 및 단일 행(Row 1) 테스트 파이프라인 정상 구동 확인
2. blocks 렌더링과 legacy_v13_monolith 렌더링 비교
3. Retrieval Pre-mapping 옵션에 따른 동작 확인
4. 생성된 결과 아티팩트(`row_comparison.csv` 등)를 로드하여 검증 결과(DET 점수, 지연 시간, 토큰 수 등) 확인

> 참고: 이 노트북은 전체 모델 비교 실험을 돌리기 위한 것이 아니라, 본격적인 대규모 실험 전 환경과 프롬프트 렌더링 파이프라인을 점검하는 용도입니다.

## Section 1. 환경 변수 및 경로 설정
제공된 가이드에 따라 Conda 가상환경 파이썬과 타겟 GPU 디바이스를 설정합니다.

In [2]:
import os
import sys
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display

# 작업 디렉토리 기준 경로 설정
NOTEBOOK_DIR = Path.cwd().resolve()
VERSION_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = VERSION_ROOT.parent.parent

# 환경 변수 설정 (로컬 환경에 맞게 수정 가능)
os.environ["JOI_V15_WORKER_PYTHON"] = "/home/mgjeong/miniconda3/envs/l/bin/python"
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"

# 실행할 파이썬 스크립트 경로
RUN_BENCHMARK_SCRIPT = VERSION_ROOT / "scripts" / "run_benchmark.py"

# 벤치마크 CLI를 실행할 Python. 보통 현재 Jupyter kernel의 Python을 사용합니다.
# 필요하면 아래 값을 직접 수정하세요. 예: "/home/mgjeong/miniconda3/envs/l/bin/python"
BENCHMARK_PYTHON = sys.executable

print("[Paths]")
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"VERSION_ROOT: {VERSION_ROOT}")
print(f"RUN_BENCHMARK_SCRIPT: {RUN_BENCHMARK_SCRIPT}")

print("\n[Environment Variables]")
print(f"WORKER_PYTHON: {os.environ.get('JOI_V15_WORKER_PYTHON')}")
print(f"LOCAL_DEVICE: {os.environ.get('JOI_V15_LOCAL_DEVICE')}")
print(f"BENCHMARK_PYTHON: {BENCHMARK_PYTHON}")

if not RUN_BENCHMARK_SCRIPT.exists():
    print(f"\n[Warning] run_benchmark.py를 찾지 못했습니다: {RUN_BENCHMARK_SCRIPT}")

[Paths]
REPO_ROOT: /root/llm/JOILang-Server
VERSION_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413
RUN_BENCHMARK_SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py

[Environment Variables]
WORKER_PYTHON: /home/mgjeong/miniconda3/envs/l/bin/python
LOCAL_DEVICE: cuda:0
BENCHMARK_PYTHON: /usr/bin/python3


## Section 2. 유틸리티 함수: 벤치마크 실행 및 최근 실행 결과 불러오기
벤치마크 스크립트 실행 후 `results/model_suite_<timestamp>` 폴더에 생성되는 CSV를 자동으로 읽어오기 위한 헬퍼 함수입니다.

In [3]:
def run_benchmark(extra_args, *, cwd=None):
    """run_benchmark.py를 subprocess로 실행합니다."""
    if not RUN_BENCHMARK_SCRIPT.exists():
        raise FileNotFoundError(f"run_benchmark.py not found: {RUN_BENCHMARK_SCRIPT}")

    cmd = [BENCHMARK_PYTHON, str(RUN_BENCHMARK_SCRIPT), *map(str, extra_args)]
    print("Running command:")
    print(" ".join(cmd))
    print()

    completed = subprocess.run(
        cmd,
        cwd=str(cwd or VERSION_ROOT),
        env=os.environ.copy(),
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(f"Benchmark failed with exit code {completed.returncode}")
    return completed


def get_latest_result_dir(version_root):
    results_dir = Path(version_root) / "results"
    if not results_dir.exists():
        return None

    subdirs = [
        d for d in results_dir.iterdir()
        if d.is_dir() and d.name.startswith("model_suite_")
    ]
    if not subdirs:
        return None

    return max(subdirs, key=lambda p: p.stat().st_mtime)


def inspect_latest_row_comparison(version_root):
    latest_dir = get_latest_result_dir(version_root)
    if latest_dir is None:
        print("No results directory found.")
        return None

    csv_path = latest_dir / "row_comparison.csv"
    if not csv_path.exists():
        print(f"row_comparison.csv not found in {latest_dir}")
        return None

    print(f"Loading results from: {csv_path}")
    df = pd.read_csv(csv_path)

    # 핵심 확인 지표 필터링
    preferred_cols = [
        "row_no",
        "model_key",
        "det_score",
        "is_pass",
        "failure_reasons",
        "prompt_tokens",
        "latency",
        "generated_code",
    ]
    target_cols = [col for col in preferred_cols if col in df.columns]
    if not target_cols:
        return df
    return df[target_cols]


## Section 3. [Recommended] 초기 Blocks Render 및 로컬 모델 테스트
가장 목적에 부합하는 권장 진입점입니다. `qwen25_coder_7b`로 Row 1에 대해 blocks 렌더링 및 Retrieval Pre-mapping 옵션을 켜서 실행합니다.

체크리스트:
1. Generated JOICode가 비어 있지 않은가?
2. JSON valid인가?
3. DET 점수는 얼마인가?
4. failure reason이 무엇인가?
5. prompt tokens가 얼마인가?
6. LLM latency가 찍히는가?

In [11]:
#MODEL="phi35_mini"

In [10]:
#MODEL="gemma2_9b_it"  #Gemma2 9B-it의 컨텍스트 한계 8192를 훨씬 넘는 20800 tokens 프롬프트

/home/mgjeong/miniconda3/envs/l/bin/python - <<'PY'
> from transformers import AutoConfig
> 
> model_path = "/home/mgjeong/.cache/huggingface/hub/models--google--gemma-2-9b-it/snapshots/11c9b309abf73637e4b6f9a3fa1e92e615547819"
> cfg = AutoConfig.from_pretrained(model_path, local_files_only=True)
> 
> for key in [
>     "max_position_embeddings",
>     "sliding_window",
>     "rope_theta",
>     "model_type",
>     "architectures",
> ]:
>     print(key, "=", getattr(cfg, key, None))
> PY

huggingface-cli login 

pip install   transformers==4.35.*   sentence-transformers==2.2.2   huggingface_hub==0.19.*



In [ ]:
#MODEL="qwen25_coder_7b"
#MODEL="llama31_8b" 
#MODEL="qwen25_coder_14b"

# Debuding 
# rm -f /tmp/joi_local_llm_worker_debug.log
# ~~
# tail -300 /tmp/joi_local_llm_worker_debug.log

In [8]:
print("Running basic block render test on qwen25_coder_14b (Row 1)...")

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "phi35_mini",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "blocks",
    "--enable-retrieval-premapping",
    "--retrieval-topk", "10",
    "--retrieval-device", "cpu",
    "--print-mode", "compare",
    "--strict-availability",
])

Running basic block render test on qwen25_coder_14b (Row 1)...
Running command:
/usr/bin/python3 /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key phi35_mini --row-no 1 --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-device cpu --print-mode compare --strict-availability

Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/model_suite_20260518_181408
Rows: 1 | Models: phi35_mini
 - phi35_mini (Phi-3.5-mini-instruct): avg_det=0.0000, gt_exact=0/1, pass=0/1, avg_latency=0.0000s, warm_p50=0.0000s, cold_load=0.0000s, avg_prompt_tokens=0.0, peak_vram=0.0000GB, gen_errors=1
   top_generation_error: failed to start persistent worker: FileNotFoundError: [Errno 2] No such file or directory: '/home/mgjeong/miniconda3/envs/l/bin/python'
Suite summary CSV: /root/llm/JOILang-Server/gpt_mg/version0_15_upd

CompletedProcess(args=['/usr/bin/python3', '/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py', '--suite', 'paper_local5', '--model-key', 'phi35_mini', '--row-no', '1', '--candidate-k', '1', '--repair-attempts', '0', '--det-profile', 'strict', '--prompt-render-mode', 'blocks', '--enable-retrieval-premapping', '--retrieval-topk', '10', '--retrieval-device', 'cpu', '--print-mode', 'compare', '--strict-availability'], returncode=0)

In [7]:
import os
from pathlib import Path

LOCAL_MODELS_DIR = Path("/root/llm/JOILang-Server/local_models").resolve()
os.environ["JOI_V15_LOCAL_MODEL_NAME"] = str(LOCAL_MODELS_DIR / "qwen25_coder_7b")
os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "qwen25_coder_7b",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "blocks",
    "--enable-retrieval-premapping",
    "--retrieval-topk", "10",
    "--retrieval-device", "cpu",
    "--print-mode", "compare",
    "--strict-availability",
])

Running command:
/usr/bin/python3 /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_7b --row-no 1 --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-device cpu --print-mode compare --strict-availability

Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/model_suite_20260518_181356
Rows: 1 | Models: qwen25_coder_7b
 - qwen25_coder_7b (Qwen2.5-Coder-7B-Instruct): avg_det=0.0000, gt_exact=0/1, pass=0/1, avg_latency=0.0000s, warm_p50=0.0000s, cold_load=0.0000s, avg_prompt_tokens=0.0, peak_vram=0.0000GB, gen_errors=1
   top_generation_error: failed to start persistent worker: FileNotFoundError: [Errno 2] No such file or directory: '/home/mgjeong/miniconda3/envs/l/bin/python'
Suite summary CSV: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/model_suite_20260518_181

CompletedProcess(args=['/usr/bin/python3', '/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py', '--suite', 'paper_local5', '--model-key', 'qwen25_coder_7b', '--row-no', '1', '--candidate-k', '1', '--repair-attempts', '0', '--det-profile', 'strict', '--prompt-render-mode', 'blocks', '--enable-retrieval-premapping', '--retrieval-topk', '10', '--retrieval-device', 'cpu', '--print-mode', 'compare', '--strict-availability'], returncode=0)

In [ ]:
# 위 셀 실행 후 생성된 결과를 데이터프레임으로 확인
df_result_1 = inspect_latest_row_comparison(VERSION_ROOT)
if df_result_1 is not None:
    display(df_result_1)

## Section 4. Full Schema Fallback (Retrieval 끄기) 테스트
Retrieval pre-mapping 없이 전체 스키마를 포함시켜 렌더링합니다. 프롬프트가 길어지므로 토큰 수와 latency 증가를 관찰할 수 있습니다.

In [6]:
print("Running Full Schema Fallback test...")

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "qwen25_coder_7b",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "blocks",
    "--disable-retrieval-premapping",
    "--print-mode", "compare",
    "--strict-availability",
])

Running Full Schema Fallback test...
Running command:
/usr/bin/python3 /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_7b --row-no 1 --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --disable-retrieval-premapping --print-mode compare --strict-availability

Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/model_suite_20260518_181350
Rows: 1 | Models: qwen25_coder_7b
 - qwen25_coder_7b (Qwen2.5-Coder-7B-Instruct): avg_det=0.0000, gt_exact=0/1, pass=0/1, avg_latency=0.0000s, warm_p50=0.0000s, cold_load=0.0000s, avg_prompt_tokens=0.0, peak_vram=0.0000GB, gen_errors=1
   top_generation_error: failed to start persistent worker: FileNotFoundError: [Errno 2] No such file or directory: '/home/mgjeong/miniconda3/envs/l/bin/python'
Suite summary CSV: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/model_suite_20260518_181350/s

CompletedProcess(args=['/usr/bin/python3', '/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py', '--suite', 'paper_local5', '--model-key', 'qwen25_coder_7b', '--row-no', '1', '--candidate-k', '1', '--repair-attempts', '0', '--det-profile', 'strict', '--prompt-render-mode', 'blocks', '--disable-retrieval-premapping', '--print-mode', 'compare', '--strict-availability'], returncode=0)

In [ ]:
df_result_full_schema = inspect_latest_row_comparison(VERSION_ROOT)
if df_result_full_schema is not None:
    display(df_result_full_schema)

## Section 5. Legacy Monolith (v13) vs Blocks (v15) 렌더링 모드 비교
`legacy_v13_monolith` 모드를 실행합니다. 이후 Section 3의 blocks 결과와 비교해볼 수 있습니다.

참고: 이 테스트를 위해서는 `version0_13` 디렉토리에 프롬프트 에셋이 존재해야 합니다.

In [ ]:
print("Running Legacy Monolith test...")

prompt_assets_dir = REPO_ROOT / "gpt_mg" / "version0_13"
print(f"prompt_assets_dir: {prompt_assets_dir}")

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "qwen25_coder_7b",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "legacy_v13_monolith",
    "--prompt-assets-dir", str(prompt_assets_dir),
    "--enable-retrieval-premapping",
    "--retrieval-topk", "10",
    "--retrieval-device", "cpu",
    "--print-mode", "compare",
    "--strict-availability",
])

In [ ]:
df_result_legacy = inspect_latest_row_comparison(VERSION_ROOT)
if df_result_legacy is not None:
    display(df_result_legacy)

## Section 6. 스케일업: 카테고리 및 여러 행 실행 예시
Row 1이 성공적으로 실행된다면, 본격적인 진화를 위해 여러 행을 평가할 수 있습니다.
아래 명령은 `--category 1`과 `--category 2`에 대해 카테고리당 3개씩, 총 6개의 데이터를 평가합니다.

실행 시간이 길어질 수 있으므로 기본적으로 주석 처리되어 있습니다. 필요 시 주석을 해제한 뒤 실행하세요.

In [ ]:
# 필요 시 아래 블록의 주석을 해제하고 실행하세요.
#
# print("Running multi-row category test...")
# run_benchmark([
#     "--suite", "paper_local5",
#     "--model-key", "qwen25_coder_7b",
#     "--category", "1",
#     "--category", "2",
#     "--limit-per-category", "3",
#     "--candidate-k", "1",
#     "--repair-attempts", "0",
#     "--det-profile", "strict",
#     "--prompt-render-mode", "blocks",
#     "--enable-retrieval-premapping",
#     "--retrieval-topk", "10",
#     "--retrieval-device", "cpu",
#     "--print-mode", "summary",
#     "--strict-availability",
# ])

print("주석을 해제하면 multi-row test를 실행할 수 있습니다.")

## Section 7. 결과 아티팩트 디렉토리 점검
가장 최근 실행된 결과 디렉토리 내에 HTML/PDF 리포트 및 각종 요약 CSV가 정상적으로 생성되었는지 확인합니다.

In [ ]:
latest_dir = get_latest_result_dir(VERSION_ROOT)
if latest_dir:
    print(f"Latest Result Directory: {latest_dir.name}\n")
    print("--- Contained Files & Reports ---")
    for item in latest_dir.rglob("*"):
        if item.is_file():
            # 너무 긴 경로는 상대 경로로 잘라서 표기
            rel_path = item.relative_to(latest_dir)
            print(f"- {rel_path}")
else:
    print("No result directory to inspect.")

# [Real Test]

## Local Inference Model 

In [1]:
from pathlib import Path
import subprocess
import os
from datetime import datetime

REPO = Path("/root/llm/JOILang-Server")
SCRIPT = REPO / "gpt_mg/version0_15_update20260413/scripts/run_benchmark.py"

os.environ["JOI_V15_WORKER_PYTHON"] = "/root/llm/je/bin/python"
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"

def run_blocks_full(model_key: str):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = REPO / f"gpt_mg/version0_15_update20260413/results/block_render_full_{model_key}_{ts}"

    cmd = [
        "python", "-u", str(SCRIPT),
        "--suite", "paper_local5",
        "--model-key", model_key,
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--prompt-render-mode", "blocks",
        "--enable-retrieval-premapping",
        "--retrieval-topk", "10",
        "--retrieval-mode", "hybrid",
        "--retrieval-device", "cpu",
        "--measure-latency",
        "--measure-vram",
        "--paper-fair-mode",
        "--export-paper-artifacts",
        "--print-mode", "summary",
        "--strict-availability",
        "--output-dir", str(out),
    ]

    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)
    return out

In [12]:
out_qwen7 = run_blocks_full("qwen25_coder_7b")

python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_7b --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-mode hybrid --retrieval-device cpu --measure-latency --measure-vram --paper-fair-mode --export-paper-artifacts --print-mode summary --strict-availability --output-dir /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_qwen25_coder_7b_20260515_160027
Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_qwen25_coder_7b_20260515_160027
Rows: 280 | Models: qwen25_coder_7b
 - qwen25_coder_7b (Qwen2.5-Coder-7B-Instruct): avg_det=79.9240, gt_exact=58/280, pass=156/280, avg_latency=4.8305s, warm_p50=5.0524s, cold_load=14.5697s, avg_prompt_tokens=27952.6, peak_vram=22.0304GB
Suite summary CSV: /root/llm/JOILang-Server/gpt_mg/ve

In [10]:
out_qwen7

PosixPath('/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_qwen25_coder_7b_20260515_145402')

In [13]:
out_llama = run_blocks_full("llama31_8b")

python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key llama31_8b --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-mode hybrid --retrieval-device cpu --measure-latency --measure-vram --paper-fair-mode --export-paper-artifacts --print-mode summary --strict-availability --output-dir /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_llama31_8b_20260515_163211
Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_llama31_8b_20260515_163211
Rows: 280 | Models: llama31_8b
 - llama31_8b (Llama-3.1-8B-Instruct): avg_det=0.0000, gt_exact=0/280, pass=0/280, avg_latency=0.0000s, warm_p50=0.0000s, cold_load=0.0000s, avg_prompt_tokens=0.0, peak_vram=0.0000GB, gen_errors=280
   top_generation_error: We couldn't connect to 'https://huggingface.co' to 

In [5]:
import os
from pathlib import Path

LOCAL_MODELS_DIR = Path("/root/llm/JOILang-Server/local_models").resolve()
os.environ["JOI_V15_LOCAL_MODEL_NAME"] = str(LOCAL_MODELS_DIR / "qwen25_coder_14b")
os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "qwen25_coder_14b",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "blocks",
    "--enable-retrieval-premapping",
    "--retrieval-topk", "10",
    "--retrieval-device", "cpu",
    "--print-mode", "compare",
    "--strict-availability",
])

Running command:
/usr/bin/python3 /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_14b --row-no 1 --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-device cpu --print-mode compare --strict-availability

Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/model_suite_20260518_181305
Rows: 1 | Models: qwen25_coder_14b
 - qwen25_coder_14b (Qwen2.5-Coder-14B-Instruct): avg_det=0.0000, gt_exact=0/1, pass=0/1, avg_latency=0.0000s, warm_p50=0.0000s, cold_load=0.0000s, avg_prompt_tokens=0.0, peak_vram=0.0000GB, gen_errors=1
   top_generation_error: failed to start persistent worker: FileNotFoundError: [Errno 2] No such file or directory: '/home/mgjeong/miniconda3/envs/l/bin/python'
Suite summary CSV: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/model_suite_20260518

CompletedProcess(args=['/usr/bin/python3', '/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py', '--suite', 'paper_local5', '--model-key', 'qwen25_coder_14b', '--row-no', '1', '--candidate-k', '1', '--repair-attempts', '0', '--det-profile', 'strict', '--prompt-render-mode', 'blocks', '--enable-retrieval-premapping', '--retrieval-topk', '10', '--retrieval-device', 'cpu', '--print-mode', 'compare', '--strict-availability'], returncode=0)

In [ ]:
out_qwen14 = run_blocks_full("qwen25_coder_14b")

python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_14b --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-mode hybrid --retrieval-device cpu --measure-latency --measure-vram --paper-fair-mode --export-paper-artifacts --print-mode summary --strict-availability --output-dir /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_qwen25_coder_14b_20260515_165037


## 2. Cloud GPT-4.1-mini blocks render 실행

cloud reference도 같은 방식으로 저장해두고 비교하려면, suite만 paper_with_cloud_ref로 바꾸고 model-key를 gpt41_mini로 지정하면 됩니다. run_benchmark.py는 실제로 run_model_suite_benchmark.py의 entrypoint를 호출하고, 그 안에 paper_with_cloud_ref = paper_local5 + GPT-4.1-mini suite가 정의

In [ ]:
from pathlib import Path
import subprocess
import os
from datetime import datetime
import pandas as pd
import json

REPO = Path("/root/llm/JOILang-Server")
SCRIPT = REPO / "gpt_mg/version0_15_update20260413/scripts/run_benchmark.py"

os.environ["JOI_V15_WORKER_PYTHON"] = "/home/mgjeong/miniconda3/envs/l/bin/python"
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"

# OpenAI-compatible endpoint가 이미 떠 있어야 합니다.
# 예: export JOI_V15_OPENAI_ENDPOINT="http://127.0.0.1:8000/v1/chat/completions"
OPENAI_ENDPOINT = os.environ.get("JOI_V15_OPENAI_ENDPOINT", "http://127.0.0.1:8000/v1/chat/completions")

In [ ]:
def run_cloud_blocks_full(
    row_no=None,
    categories=None,
    limit_per_category=None,
    output_prefix="cloud_blocks_full",
):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = REPO / f"gpt_mg/version0_15_update20260413/results/{output_prefix}_gpt41_mini_{ts}"

    cmd = [
        "python", "-u", str(SCRIPT),
        "--suite", "paper_with_cloud_ref",
        "--model-key", "gpt41_mini",
        "--llm-mode", "openai",
        "--llm-endpoint", OPENAI_ENDPOINT,
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--prompt-render-mode", "blocks",
        "--enable-retrieval-premapping",
        "--retrieval-topk", "10",
        "--retrieval-mode", "hybrid",
        "--retrieval-device", "cpu",
        "--print-mode", "summary",
        "--output-dir", str(out),
    ]

    if row_no is not None:
        cmd += ["--row-no", str(row_no)]

    if categories:
        for cat in categories:
            cmd += ["--category", str(cat)]

    if limit_per_category is not None:
        cmd += ["--limit-per-category", str(limit_per_category)]

    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)

    return out

In [ ]:
# Row 1만 cloud blocks render로 확인
out_cloud_row1 = run_cloud_blocks_full(row_no=1, output_prefix="cloud_blocks_row1")

In [ ]:
# 전체 dataset cloud blocks render
out_cloud_full = run_cloud_blocks_full(output_prefix="cloud_blocks_full")

In [ ]:
# category별 일부만 먼저 확인
out_cloud_cat12 = run_cloud_blocks_full(
    categories=[1, 2],
    limit_per_category=5,
    output_prefix="cloud_blocks_cat12"
)

### 3. 결과확인

In [ ]:
import pandas as pd
from pathlib import Path

def inspect_result(out_dir):
    out_dir = Path(out_dir)

    print("OUT:", out_dir)

    files = [
        "suite_summary.csv",
        "main_model_comparison.csv",
        "tradeoff_summary.csv",
        "latency_summary.csv",
        "vram_summary.csv",
        "tokenizer_summary.csv",
        "paper_metrics_summary.json",
        "row_comparison.csv",
        "failure_reason_summary.csv",
        "category_summary.csv",
    ]

    for f in files:
        p = out_dir / f
        print(f, "OK" if p.exists() else "MISSING")

    summary = out_dir / "suite_summary.csv"
    if summary.exists():
        display(pd.read_csv(summary))

    main = out_dir / "main_model_comparison.csv"
    if main.exists():
        display(pd.read_csv(main))

    failure = out_dir / "failure_reason_summary.csv"
    if failure.exists():
        display(pd.read_csv(failure).head(20))

    category = out_dir / "category_summary.csv"
    if category.exists():
        display(pd.read_csv(category))

In [ ]:
# Cloud legacy monolith도 비교하고 싶을 때
def run_cloud_legacy_full(
    row_no=None,
    categories=None,
    limit_per_category=None,
    output_prefix="cloud_legacy_full",
):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = REPO / f"gpt_mg/version0_15_update20260413/results/{output_prefix}_gpt41_mini_{ts}"

    prompt_assets = REPO / "gpt_mg/version0_13"

    cmd = [
        "python", "-u", str(SCRIPT),
        "--suite", "paper_with_cloud_ref",
        "--model-key", "gpt41_mini",
        "--llm-mode", "openai",
        "--llm-endpoint", OPENAI_ENDPOINT,
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--prompt-render-mode", "legacy_v13_monolith",
        "--prompt-assets-dir", str(prompt_assets),
        "--enable-retrieval-premapping",
        "--retrieval-topk", "10",
        "--retrieval-mode", "hybrid",
        "--retrieval-device", "cpu",
        "--print-mode", "summary",
        "--output-dir", str(out),
    ]

    if row_no is not None:
        cmd += ["--row-no", str(row_no)]

    if categories:
        for cat in categories:
            cmd += ["--category", str(cat)]

    if limit_per_category is not None:
        cmd += ["--limit-per-category", str(limit_per_category)]

    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)

    return out

In [ ]:
out_cloud_legacy_row1 = run_cloud_legacy_full(row_no=1, output_prefix="cloud_blocks_full")

In [ ]:
# 결과를 Jupyter에서 바로 읽기

def load_result_tables(out_dir):
    out_dir = Path(out_dir)

    files = {
        "suite_summary": out_dir / "suite_summary.csv",
        "row_comparison": out_dir / "row_comparison.csv",
        "failure_reason_summary": out_dir / "failure_reason_summary.csv",
        "category_summary": out_dir / "category_summary.csv",
        "main_model_comparison": out_dir / "main_model_comparison.csv",
        "tradeoff_summary": out_dir / "tradeoff_summary.csv",
        "paper_metrics_summary": out_dir / "paper_metrics_summary.json",
    }

    loaded = {}
    for name, path in files.items():
        if path.exists():
            if path.suffix == ".csv":
                loaded[name] = pd.read_csv(path)
            elif path.suffix == ".json":
                loaded[name] = json.loads(path.read_text(encoding="utf-8"))
        else:
            print(f"[MISSING] {name}: {path}")

    return loaded

In [ ]:
cloud_tables = load_result_tables(out_cloud_full)

display(cloud_tables["suite_summary"])
display(cloud_tables["category_summary"])
display(cloud_tables["failure_reason_summary"].head(20))

In [ ]:
# Cloud blocks vs cloud legacy 비교
out_cloud_blocks_row1 = run_cloud_blocks_full(row_no=1, output_prefix="cloud_blocks_row1")
out_cloud_legacy_row1 = run_cloud_legacy_full(row_no=1, output_prefix="cloud_legacy_row1")

cmp = pd.DataFrame([
    summarize_run(out_cloud_blocks_row1, "gpt41_blocks_row1"),
    summarize_run(out_cloud_legacy_row1, "gpt41_legacy_row1"),
])
display(cmp)

### Local 결과와 Cloud 결과 비교

In [ ]:
out_qwen7 = run_blocks_full("qwen25_coder_7b")
out_cloud_full = run_cloud_blocks_full()

In [ ]:
def summarize_run(out_dir, label):
    tables = load_result_tables(out_dir)
    suite = tables.get("suite_summary")
    if suite is None:
        return None

    row = suite.iloc[0].to_dict()
    row["run_label"] = label
    row["out_dir"] = str(out_dir)
    return row

summary_rows = []
summary_rows.append(summarize_run(out_qwen7, "qwen7_blocks"))
summary_rows.append(summarize_run(out_cloud_full, "gpt41_blocks"))

summary_df = pd.DataFrame([r for r in summary_rows if r is not None])
display(summary_df)

In [ ]:
# 1) Cloud blocks row 1 smoke
out_cloud_row1 = run_cloud_blocks_full(row_no=1, output_prefix="cloud_blocks_row1")

# 2) Local qwen7 row 1 smoke
out_qwen7_row1 = run_blocks_full("qwen25_coder_7b", row_no=1)

# 3) Cloud blocks category 일부
out_cloud_cat12 = run_cloud_blocks_full(categories=[1, 2], limit_per_category=5)

# 4) Local qwen7 category 일부
out_qwen7_cat12 = run_blocks_full("qwen25_coder_7b", categories=[1, 2], limit_per_category=5)

# 5) 문제 없으면 cloud full
out_cloud_full = run_cloud_blocks_full(output_prefix="cloud_blocks_full")

# 6) 문제 없으면 local full
out_qwen7_full = run_blocks_full("qwen25_coder_7b", output_prefix="qwen7_blocks_full")

In [9]:
from pathlib import Path
import subprocess
import os
import sys
from datetime import datetime

REPO = Path("/root/llm/JOILang-Server")
SCRIPT_BENCH = REPO / "gpt_mg/version0_15_update20260413/scripts/run_benchmark.py"

os.environ["JOI_V15_WORKER_PYTHON"] = "/root/llm/je/bin/python"
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"

def run_blocks_smoke(
    model_key: str,
    row_no: int | None = 1,
    categories: list[int] | None = None,
    limit_per_category: int | None = None,
    out_prefix: str = "block_render_smoke",
):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = REPO / f"gpt_mg/version0_15_update20260413/results/{out_prefix}_{model_key}_{ts}"

    cmd = [
        sys.executable, "-u", str(SCRIPT_BENCH),
        "--suite", "paper_local5",
        "--model-key", model_key,
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--prompt-render-mode", "blocks",
        "--enable-retrieval-premapping",
        "--retrieval-topk", "10",
        "--retrieval-mode", "hybrid",
        "--retrieval-device", "cpu",
        "--print-mode", "compare" if row_no is not None else "summary",
        "--strict-availability",
        "--output-dir", str(out),
    ]

    if row_no is not None:
        cmd += ["--row-no", str(row_no)]

    if categories:
        for c in categories:
            cmd += ["--category", str(c)]

    if limit_per_category is not None:
        cmd += ["--limit-per-category", str(limit_per_category)]

    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)
    return out

In [10]:
out_smoke = run_blocks_smoke("qwen25_coder_14b", row_no=1)

/usr/bin/python3 -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_14b --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-mode hybrid --retrieval-device cpu --print-mode compare --strict-availability --output-dir /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_smoke_qwen25_coder_14b_20260518_181840 --row-no 1
Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_smoke_qwen25_coder_14b_20260518_181840
Rows: 1 | Models: qwen25_coder_14b
 - qwen25_coder_14b (Qwen2.5-Coder-14B-Instruct): avg_det=100.0000, gt_exact=1/1, pass=1/1, avg_latency=25.7745s, warm_p50=25.7745s, cold_load=0.0000s, avg_prompt_tokens=41926.0, peak_vram=22.0131GB
Suite summary CSV: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_smoke_qw

In [12]:
out_cat12 = run_blocks_smoke("qwen25_coder_14b", row_no=None, categories=[1, 2], limit_per_category=2)

/usr/bin/python3 -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_14b --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-mode hybrid --retrieval-device cpu --print-mode summary --strict-availability --output-dir /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_smoke_qwen25_coder_14b_20260518_182025 --category 1 --category 2 --limit-per-category 2
Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_smoke_qwen25_coder_14b_20260518_182025
Rows: 4 | Models: qwen25_coder_14b
 - qwen25_coder_14b (Qwen2.5-Coder-14B-Instruct): avg_det=92.4750, gt_exact=3/4, pass=3/4, avg_latency=6.3162s, warm_p50=4.3036s, cold_load=0.0000s, avg_prompt_tokens=28136.0, peak_vram=22.0146GB
Suite summary CSV: /root/llm/JOILang-Server/gpt_mg/version0_15_update202

### GA 결과를 Jupyter에서 확인하는 코드

ga_progress_all.csv 같은 그래프를 만들려면 필요한 파일

Figure 1을 제대로 그리려면 최소한 아래 파일이 필요합니다.

ga_generation_progress.csv

ga_topk_genomes.csv

ga_block_diffs.jsonl

ga_population_diagnostics.csv

structured_feedback.jsonl

ga_summary.json

네가 올린 ga_progress_all.csv는 형식상 Figure 1의 재료가 될 수는 있지만, 현재 값이 전부 0이라 논문용 그래프로는 불가합니다.

In [16]:
out_ga_qwen14 = run_ga_category_smoke(
    model_key="qwen25_coder_14b",
    categories=[1, 2],
    limit_per_category=2,
    population=4,
    gens=2,
    out_prefix="ga_real_cat12",
    use_advisor=False,
)

/usr/bin/python3 -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_14b --population 4 --gens 2 --sample-size 4 --validation-size 4 --cheap-eval-limit 2 --candidate-k 1 --repair-attempts 0 --det-profile strict --feedback-guided-mutation --progress minimal --output-root /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_real_cat12_qwen25_coder_14b_20260518_182407 --category 1 --category 2 --limit-per-category 2
[RUN]
command=/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_14b --population 4 --gens 2 --sample-size 4 --validation-size 4 --cheap-eval-limit 2 --candidate-k 1 --repair-attempts 0 --det-profile strict --feedback-guided-mutation --progress minimal --output-root /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_real_cat12_qwen25_coder_14b_20260518_182407 --category 1 --category

In [ ]:
import pandas as pd
from pathlib import Path
import json

def inspect_ga_result(out_dir):
    out_dir = Path(out_dir)
    print("OUT:", out_dir)

    files = [
        "ga_generation_progress.csv",
        "ga_generation_progress.jsonl",
        "ga_topk_genomes.csv",
        "ga_population_diagnostics.csv",
        "structured_feedback_summary.csv",
        "population_transitions.csv",
        "ga_summary.json",
    ]

    for f in files:
        print(f, "OK" if (out_dir / f).exists() else "MISSING")

    progress = out_dir / "ga_generation_progress.csv"
    if progress.exists():
        df = pd.read_csv(progress)
        print("\n[ga_generation_progress columns]")
        print(list(df.columns))
        display(df.head(20))

        metric_candidates = [
            "train_avg_det_score",
            "validation_avg_det_score",
            "train_det_pass_rate",
            "validation_det_pass_rate",
            "avg_det_score",
            "det_pass_rate",
            "fitness",
        ]

        available_metrics = [c for c in metric_candidates if c in df.columns]

        if {"model_key", "generation"}.issubset(df.columns) and available_metrics:
            display(
                df.groupby(["model_key", "generation"])[available_metrics]
                  .mean(numeric_only=True)
                  .reset_index()
            )
        else:
            print("\n[WARN] This progress file does not contain evaluated GA metric columns.")
            print("Likely dry-run or incomplete run.")

    topk = out_dir / "ga_topk_genomes.csv"
    if topk.exists():
        print("\n[top-k genomes]")
        topk_df = pd.read_csv(topk)
        print(list(topk_df.columns))
        display(topk_df.head(20))

    diag = out_dir / "ga_population_diagnostics.csv"
    if diag.exists():
        print("\n[population diagnostics]")
        diag_df = pd.read_csv(diag)
        print(list(diag_df.columns))
        display(diag_df.head(20))

    summary = out_dir / "ga_summary.json"
    if summary.exists():
        print("\n[summary]")
        print(json.dumps(json.loads(summary.read_text()), indent=2, ensure_ascii=False)[:4000])

In [19]:
inspect_ga_result(out_ga_qwen14)

OUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_real_cat12_qwen25_coder_14b_20260518_182407
ga_generation_progress.csv OK
ga_topk_genomes.csv OK
ga_population_diagnostics.csv OK
structured_feedback_summary.csv OK
population_transitions.csv OK
ga_summary.json OK


,generation,model_key,genome_id


KeyError: "Columns not found: 'validation_det_pass_rate', 'train_det_pass_rate', 'train_avg_det_score', 'validation_avg_det_score'"

### Figure 1용 그래프를 그리는 코드

In [20]:
import matplotlib.pyplot as plt

def plot_ga_generation_curve(out_dir):
    out_dir = Path(out_dir)
    progress = out_dir / "ga_generation_progress.csv"
    if not progress.exists():
        raise FileNotFoundError(progress)

    df = pd.read_csv(progress)

    # 컬럼명 방어
    y_col = None
    for c in ["validation_det_pass_rate", "train_det_pass_rate", "validation_avg_det_score", "train_avg_det_score"]:
        if c in df.columns:
            y_col = c
            break

    if y_col is None:
        raise ValueError("No DET/pass metric column found")

    plt.figure(figsize=(8, 4.5))
    for model_key, sub in df.groupby("model_key"):
        curve = sub.groupby("generation")[y_col].mean().reset_index()
        plt.plot(curve["generation"], curve[y_col], marker="o", label=model_key)

    plt.xlabel("Generation")
    plt.ylabel(y_col)
    plt.title("GA search dynamics")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

def plot_ga_dynamics(out_dir):
    out_dir = Path(out_dir)
    progress = out_dir / "ga_generation_progress.csv"
    if not progress.exists():
        raise FileNotFoundError(progress)

    df = pd.read_csv(progress)
    print("columns:", list(df.columns))

    y_col = None
    for c in [
        "validation_det_pass_rate",
        "train_det_pass_rate",
        "det_pass_rate",
        "validation_avg_det_score",
        "train_avg_det_score",
        "avg_det_score",
        "fitness",
    ]:
        if c in df.columns:
            y_col = c
            break

    if y_col is None:
        print("[WARN] No DET/pass metric column found. Cannot draw meaningful GA dynamics.")
        return

    if "generation" not in df.columns:
        print("[WARN] No generation column found.")
        return

    plt.figure(figsize=(8, 4.5))

    if "model_key" in df.columns:
        for model_key, sub in df.groupby("model_key"):
            curve = sub.groupby("generation")[y_col].mean().reset_index()
            plt.plot(curve["generation"], curve[y_col], marker="o", label=model_key)
        plt.legend()
    else:
        curve = df.groupby("generation")[y_col].mean().reset_index()
        plt.plot(curve["generation"], curve[y_col], marker="o")

    plt.xlabel("Generation")
    plt.ylabel(y_col)
    plt.title("GA search dynamics")
    plt.grid(True, alpha=0.3)
    plt.show()

In [21]:
plot_ga_generation_curve(out_ga_qwen14)

ValueError: No DET/pass metric column found

In [23]:
SCRIPT = REPO / "gpt_mg/version0_15_update20260413/scripts/run_benchmark.py"

In [44]:
from pathlib import Path
import subprocess
import os
import sys
import json
import time
import queue
import threading
import traceback
from datetime import datetime

REPO = Path("/root/llm/JOILang-Server")
SCRIPT_GA = REPO / "gpt_mg/version0_15_update20260413/scripts/run_ga_search.py"
RESULTS_ROOT = REPO / "gpt_mg/version0_15_update20260413/results"

os.environ["JOI_V15_WORKER_PYTHON"] = "/root/llm/je/bin/python"
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"


def make_ga_run_name(
    model_key: str,
    limit_per_category: int,
    population: int,
    gens: int,
    categories,
    base: str = "ga_allcat",
) -> str:
    cats = list(categories)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    if cats == list(range(1, 9)):
        cat_part = f"allcat{limit_per_category}"
    else:
        cat_text = "-".join(str(c) for c in cats)
        cat_part = f"cat{cat_text}_lpc{limit_per_category}"

    return f"{base}_{cat_part}_pop{population}_gens{gens}_{model_key}_{ts}"


def _latest_mtime(root: Path) -> float:
    if not root.exists():
        return 0.0

    latest = 0.0
    for p in root.rglob("*"):
        try:
            if p.is_file():
                latest = max(latest, p.stat().st_mtime)
        except FileNotFoundError:
            pass
    return latest


def _reader_thread(pipe, q: queue.Queue):
    try:
        for line in iter(pipe.readline, ""):
            q.put(line)
    finally:
        try:
            pipe.close()
        except Exception:
            pass


def run_command_with_watchdog(
    cmd: list[str],
    cwd: Path,
    ga_output_dir: Path,
    watchdog_dir: Path,
    idle_timeout_sec: int = 1800,
    total_timeout_sec: int | None = None,
):
    """
    watchdog_dir can be inside the run folder.
    ga_output_dir must remain empty before run_ga_search.py starts.
    """

    watchdog_dir.mkdir(parents=True, exist_ok=True)

    log_path = watchdog_dir / "run_stdout_stderr.log"
    status_path = watchdog_dir / "run_watchdog_status.json"
    command_path = watchdog_dir / "command_used.json"

    command_path.write_text(
        json.dumps(
            {
                "cmd": cmd,
                "cwd": str(cwd),
                "ga_output_dir": str(ga_output_dir),
                "watchdog_dir": str(watchdog_dir),
                "start_time": datetime.now().isoformat(timespec="seconds"),
                "idle_timeout_sec": idle_timeout_sec,
                "total_timeout_sec": total_timeout_sec,
            },
            ensure_ascii=False,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    q = queue.Queue()
    t = threading.Thread(target=_reader_thread, args=(proc.stdout, q), daemon=True)
    t.start()

    start = time.time()
    last_activity = time.time()
    last_artifact_mtime = _latest_mtime(ga_output_dir)

    def write_status(status: str, reason: str = ""):
        status_path.write_text(
            json.dumps(
                {
                    "status": status,
                    "reason": reason,
                    "returncode": proc.poll(),
                    "elapsed_sec": round(time.time() - start, 2),
                    "last_activity_age_sec": round(time.time() - last_activity, 2),
                    "timestamp": datetime.now().isoformat(timespec="seconds"),
                    "log_path": str(log_path),
                    "watchdog_dir": str(watchdog_dir),
                    "ga_output_dir": str(ga_output_dir),
                },
                ensure_ascii=False,
                indent=2,
            )
            + "\n",
            encoding="utf-8",
        )

    write_status("RUNNING")

    try:
        with log_path.open("a", encoding="utf-8") as log:
            log.write("[COMMAND]\n")
            log.write(" ".join(cmd) + "\n\n")
            log.flush()

            while True:
                while True:
                    try:
                        line = q.get_nowait()
                    except queue.Empty:
                        break

                    last_activity = time.time()
                    print(line, end="")
                    log.write(line)
                    log.flush()

                current_mtime = _latest_mtime(ga_output_dir)
                if current_mtime > last_artifact_mtime:
                    last_artifact_mtime = current_mtime
                    last_activity = time.time()

                rc = proc.poll()
                if rc is not None:
                    while True:
                        try:
                            line = q.get_nowait()
                        except queue.Empty:
                            break
                        print(line, end="")
                        log.write(line)
                        log.flush()

                    if rc == 0:
                        write_status("PASS")
                        return

                    write_status("FAILED", f"process exited with non-zero return code: {rc}")
                    raise subprocess.CalledProcessError(rc, cmd)

                idle = time.time() - last_activity
                if idle_timeout_sec is not None and idle > idle_timeout_sec:
                    reason = (
                        f"idle timeout: no stdout/stderr and no artifact update "
                        f"for {idle_timeout_sec} sec"
                    )
                    write_status("TIMEOUT_IDLE", reason)

                    proc.terminate()
                    try:
                        proc.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        proc.kill()
                        proc.wait(timeout=30)

                    raise TimeoutError(reason)

                elapsed = time.time() - start
                if total_timeout_sec is not None and elapsed > total_timeout_sec:
                    reason = f"total timeout: exceeded {total_timeout_sec} sec"
                    write_status("TIMEOUT_TOTAL", reason)

                    proc.terminate()
                    try:
                        proc.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        proc.kill()
                        proc.wait(timeout=30)

                    raise TimeoutError(reason)

                time.sleep(2)

    except Exception as exc:
        error_path = watchdog_dir / "run_error.json"
        error_path.write_text(
            json.dumps(
                {
                    "status": "ERROR",
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                    "traceback": traceback.format_exc(),
                    "timestamp": datetime.now().isoformat(timespec="seconds"),
                    "log_path": str(log_path),
                    "watchdog_dir": str(watchdog_dir),
                    "ga_output_dir": str(ga_output_dir),
                },
                ensure_ascii=False,
                indent=2,
            )
            + "\n",
            encoding="utf-8",
        )
        raise


def run_ga_all_categories(
    model_key: str,
    categories=range(1, 9),
    limit_per_category: int = 3,
    population: int = 6,
    gens: int = 3,
    base_prefix: str = "ga",
    use_advisor: bool = False,
    llm_mode: str | None = None,
    full_run: bool = True,
    progress: str = "verbose",
    timeout_sec: int = 600,
    retries: int = 0,
    idle_timeout_sec: int = 1800,
    total_timeout_sec: int | None = None,
):
    cats = list(categories)

    run_name = make_ga_run_name(
        model_key=model_key,
        limit_per_category=limit_per_category,
        population=population,
        gens=gens,
        categories=cats,
        base=base_prefix,
    )

    run_root = RESULTS_ROOT / run_name
    ga_output_dir = run_root / "ga_output"
    watchdog_dir = run_root / "_watchdog_logs"

    run_root.mkdir(parents=True, exist_ok=False)

    cmd = [
        sys.executable,
        "-u",
        str(SCRIPT_GA),
        "--profile",
        "version0_15",
        "--model-key",
        model_key,
        "--population",
        str(population),
        "--gens",
        str(gens),
        "--sample-size",
        str(limit_per_category * len(cats)),
        "--validation-size",
        str(limit_per_category * len(cats)),
        "--cheap-eval-limit",
        "2",
        "--candidate-k",
        "1",
        "--repair-attempts",
        "0",
        "--det-profile",
        "strict",
        "--feedback-guided-mutation",
        "--progress",
        progress,
        "--timeout-sec",
        str(timeout_sec),
        "--retries",
        str(retries),
        "--output-root",
        str(ga_output_dir),
    ]

    for c in cats:
        cmd += ["--category", str(c)]

    cmd += ["--limit-per-category", str(limit_per_category)]

    if full_run:
        cmd += ["--full-run"]

    if use_advisor:
        cmd += ["--llm-mutation-advisor", "--advisor-model-key", "gpt41_mini"]
        if llm_mode:
            cmd += ["--llm-mode", llm_mode]

    layout_path = run_root / "run_layout.json"
    layout_path.write_text(
        json.dumps(
            {
                "run_name": run_name,
                "run_root": str(run_root),
                "ga_output_dir": str(ga_output_dir),
                "watchdog_dir": str(watchdog_dir),
                "model_key": model_key,
                "categories": cats,
                "limit_per_category": limit_per_category,
                "population": population,
                "gens": gens,
                "sample_size": limit_per_category * len(cats),
                "validation_size": limit_per_category * len(cats),
            },
            ensure_ascii=False,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

    print("RUN_ROOT:", run_root)
    print("GA_OUTPUT:", ga_output_dir)
    print("WATCHDOG:", watchdog_dir)
    print("COMMAND:", " ".join(cmd))

    run_command_with_watchdog(
        cmd=cmd,
        cwd=REPO,
        ga_output_dir=ga_output_dir,
        watchdog_dir=watchdog_dir,
        idle_timeout_sec=idle_timeout_sec,
        total_timeout_sec=total_timeout_sec,
    )

    return ga_output_dir

In [38]:
from pathlib import Path
import subprocess
import os
import sys
from datetime import datetime

REPO = Path("/root/llm/JOILang-Server")
SCRIPT_GA = REPO / "gpt_mg/version0_15_update20260413/scripts/run_ga_search.py"

os.environ["JOI_V15_WORKER_PYTHON"] = "/root/llm/je/bin/python"
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"

def run_ga_category_smoke(
    model_key: str,
    categories: list[int] = [1, 2],
    limit_per_category: int = 2,
    population: int = 4,
    gens: int = 2,
    out_prefix: str = "ga_smoke",
    use_advisor: bool = False,
    llm_mode: str | None = None,
):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = REPO / f"gpt_mg/version0_15_update20260413/results/{out_prefix}_{model_key}_{ts}"

    cmd = [
        sys.executable, "-u", str(SCRIPT_GA),
        "--profile", "version0_15",
        "--model-key", model_key,
        "--population", str(population),
        "--gens", str(gens),
        "--sample-size", str(limit_per_category * len(categories)),
        "--validation-size", str(limit_per_category * len(categories)),
        "--cheap-eval-limit", "2",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--feedback-guided-mutation",
        "--progress", "minimal",
        "--output-root", str(out),
    ]

    for c in categories:
        cmd += ["--category", str(c)]

    cmd += ["--limit-per-category", str(limit_per_category)]

    if use_advisor:
        cmd += ["--llm-mutation-advisor", "--advisor-model-key", "gpt41_mini"]
        if llm_mode:
            cmd += ["--llm-mode", llm_mode]

    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)
    return out

In [43]:
import inspect
print(inspect.getsource(run_command_with_watchdog).splitlines()[:25])

['def run_command_with_watchdog(', '    cmd: list[str],', '    cwd: Path,', '    out_dir: Path,', '    idle_timeout_sec: int = 1800,', '    total_timeout_sec: int | None = None,', '):', '    """', '    Watchdog files are written outside GA output-root.', '', '    Do not create/write files in out_dir before run_ga_search.py starts,', '    because run_ga_search.py rejects non-empty output directories unless', '    --force or --resume is used.', '    """', '', '    watchdog_dir = out_dir.parent / "_watchdog_logs" / out_dir.name', '    watchdog_dir.mkdir(parents=True, exist_ok=True)', '', '    log_path = watchdog_dir / "run_stdout_stderr.log"', '    status_path = watchdog_dir / "run_watchdog_status.json"', '    command_path = watchdog_dir / "command_used.json"', '', '    command_path.write_text(', '        json.dumps(', '            {']


In [ ]:
out_ga_qwen14_allcat = run_ga_all_categories(
    model_key="qwen25_coder_14b",
    categories=range(1, 9),
    limit_per_category=3,
    population=5,
    gens=10,
    base_prefix="ga",
    use_advisor=False,
    full_run=True,
    progress="verbose",
    timeout_sec=600,
    retries=0,
    idle_timeout_sec=2400,
    total_timeout_sec=12 * 3600,
)

RUN_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_allcat3_pop5_gens10_qwen25_coder_14b_20260518_222332
GA_OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_allcat3_pop5_gens10_qwen25_coder_14b_20260518_222332/ga_output
WATCHDOG: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_allcat3_pop5_gens10_qwen25_coder_14b_20260518_222332/_watchdog_logs
COMMAND: /usr/bin/python3 -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_14b --population 5 --gens 10 --sample-size 24 --validation-size 24 --cheap-eval-limit 2 --candidate-k 1 --repair-attempts 0 --det-profile strict --feedback-guided-mutation --progress verbose --timeout-sec 600 --retries 0 --output-root /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_allcat3_pop5_gens10_qwen25_coder_14b_20260518_222332/ga_output --category 1 --category 2 --category 3 --catego

In [ ]:
plot_ga_dynamics(out_ga_qwen14_allcat)

In [31]:
plot_ga_dynamics(out_ga_qwen14_allcat)

NameError: name 'plot_ga_dynamics' is not defined

In [32]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec


def _to_percent(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    if s.dropna().max() is not None and s.dropna().max() <= 1.000001:
        return s * 100.0
    return s


def _pick_y_col(df: pd.DataFrame) -> str:
    for c in [
        "validation_det_pass_rate",
        "det_pass_rate",
        "DETPass",
        "mean_DETPass",
        "validation_avg_det_score",
        "avg_det_score",
        "fitness",
    ]:
        if c in df.columns:
            return c
    raise ValueError(f"No usable DET/pass metric column found. columns={list(df.columns)}")


def _load_generation_progress(out_dirs: dict[str, str | Path]) -> pd.DataFrame:
    frames = []

    for label, out_dir in out_dirs.items():
        out_dir = Path(out_dir)

        candidates = [
            out_dir / "ga_generation_progress.csv",
            out_dir / "paper" / "figure_data" / "figure1_generation_dynamics.csv",
            out_dir / "figure_data" / "figure1_generation_dynamics.csv",
        ]

        path = next((p for p in candidates if p.exists()), None)
        if path is None:
            print(f"[WARN] no generation progress csv for {label}: {out_dir}")
            continue

        df = pd.read_csv(path)
        df["model_label_for_plot"] = label
        if "model_key" not in df.columns:
            df["model_key"] = label
        frames.append(df)

    if not frames:
        raise FileNotFoundError("No ga_generation_progress.csv or figure1_generation_dynamics.csv found.")

    return pd.concat(frames, ignore_index=True)


def _load_category_diagnostics(out_dirs: dict[str, str | Path]) -> pd.DataFrame:
    frames = []

    for label, out_dir in out_dirs.items():
        out_dir = Path(out_dir)

        candidates = [
            out_dir / "ga_population_diagnostics.csv",
            out_dir / "paper" / "figure_data" / "figure1_category_dynamics.csv",
            out_dir / "figure_data" / "figure1_category_dynamics.csv",
        ]

        path = next((p for p in candidates if p.exists()), None)
        if path is None:
            print(f"[WARN] no category diagnostic csv for {label}: {out_dir}")
            continue

        df = pd.read_csv(path)
        df["model_label_for_plot"] = label
        if "model_key" not in df.columns:
            df["model_key"] = label
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


def _category_group(category) -> str:
    token = str(category).strip()

    if token in {"1", "2"}:
        return "Basic"
    if token in {"3", "4", "5"}:
        return "Temporal"
    if token in {"6", "7", "8"}:
        return "Complex"

    return f"Category {token}"


def _weighted_group_mean(df: pd.DataFrame, y_col: str) -> pd.DataFrame:
    df = df.copy()
    df[y_col] = _to_percent(df[y_col])

    if "row_evaluations" in df.columns:
        df["row_evaluations"] = pd.to_numeric(df["row_evaluations"], errors="coerce").fillna(1.0)

        rows = []
        for keys, sub in df.groupby(["model_label_for_plot", "group", "generation"], dropna=False):
            weight = sub["row_evaluations"]
            y = sub[y_col]
            if weight.sum() > 0:
                mean_y = (y * weight).sum() / weight.sum()
            else:
                mean_y = y.mean()
            rows.append(
                {
                    "model_label_for_plot": keys[0],
                    "group": keys[1],
                    "generation": keys[2],
                    y_col: mean_y,
                }
            )
        return pd.DataFrame(rows)

    return (
        df.groupby(["model_label_for_plot", "group", "generation"], as_index=False)[y_col]
        .mean()
    )


def plot_paper_ga_dynamics(
    out_dirs: dict[str, str | Path],
    save_path: str | Path | None = None,
    title: str = "Figure 1. Prompt Search Dynamics Across Model Scales",
    hand_crafted_baseline: float | None = 25.0,
    cloud_transfer_baseline: float | None = 50.0,
):
    """
    out_dirs example:
        {
            "4B": "/path/to/ga_allcat_phi35_mini_...",
            "7B": "/path/to/ga_allcat_qwen25_coder_7b_...",
            "14B": "/path/to/ga_allcat_qwen25_coder_14b_...",
        }

    Required for left panel:
        ga_generation_progress.csv

    Required for right category panels:
        ga_population_diagnostics.csv
        or figure1_category_dynamics.csv
    """

    gen_df = _load_generation_progress(out_dirs)
    y_col = _pick_y_col(gen_df)

    if "generation" not in gen_df.columns:
        raise ValueError("generation column is required.")

    gen_df = gen_df.copy()
    gen_df["generation"] = pd.to_numeric(gen_df["generation"], errors="coerce")
    gen_df[y_col] = _to_percent(gen_df[y_col])

    overall = (
        gen_df.groupby(["model_label_for_plot", "generation"], as_index=False)[y_col]
        .mean()
        .dropna(subset=["generation", y_col])
    )

    cat_df = _load_category_diagnostics(out_dirs)

    fig = plt.figure(figsize=(15, 9))
    gs = GridSpec(
        nrows=3,
        ncols=2,
        width_ratios=[3.2, 1.25],
        height_ratios=[1, 1, 1],
        wspace=0.15,
        hspace=0.5,
    )

    ax_main = fig.add_subplot(gs[:, 0])
    ax_basic = fig.add_subplot(gs[0, 1])
    ax_temporal = fig.add_subplot(gs[1, 1])
    ax_complex = fig.add_subplot(gs[2, 1])

    axes_by_group = {
        "Basic": ax_basic,
        "Temporal": ax_temporal,
        "Complex": ax_complex,
    }

    # Left main panel
    for label, sub in overall.groupby("model_label_for_plot"):
        sub = sub.sort_values("generation")
        ax_main.plot(sub["generation"], sub[y_col], linewidth=2.5, label=label)

    if hand_crafted_baseline is not None:
        ax_main.axhline(hand_crafted_baseline, linestyle="--", linewidth=1.5, label="Hand-crafted baseline")

    if cloud_transfer_baseline is not None:
        ax_main.axhline(cloud_transfer_baseline, linestyle="--", linewidth=1.5, label="Cloud-transfer baseline")

    ax_main.set_xlabel("GA Generation", fontsize=14)
    ax_main.set_ylabel("DETPass (%)", fontsize=14)
    ax_main.set_ylim(0, 100)
    ax_main.grid(True, alpha=0.35)
    ax_main.legend(loc="lower right", fontsize=10)

    # Optional paper-style annotations
    ax_main.annotate(
        "Fast early gains",
        xy=(8, 73),
        xytext=(4, 85),
        arrowprops={"arrowstyle": "->", "linewidth": 1.0},
        fontsize=11,
        fontstyle="italic",
    )

    ax_main.annotate(
        "Late crossover",
        xy=(34, 82),
        xytext=(36, 72),
        arrowprops={"arrowstyle": "->", "linewidth": 1.0},
        fontsize=11,
        fontstyle="italic",
    )

    # Right category panels
    if not cat_df.empty and "generation" in cat_df.columns and "category" in cat_df.columns:
        cat_df = cat_df.copy()
        cat_df["generation"] = pd.to_numeric(cat_df["generation"], errors="coerce")
        cat_df["group"] = cat_df["category"].map(_category_group)

        cat_y = _pick_y_col(cat_df)
        cat_grouped = _weighted_group_mean(cat_df, cat_y).dropna(subset=["generation", cat_y])

        for group_name, ax in axes_by_group.items():
            group_data = cat_grouped[cat_grouped["group"] == group_name]

            for label, sub in group_data.groupby("model_label_for_plot"):
                sub = sub.sort_values("generation")
                ax.plot(sub["generation"], sub[cat_y], linewidth=2.0)

            ax.set_title(group_name, fontsize=13)
            ax.set_xlabel("GA Generation", fontsize=10)
            ax.set_ylabel("DETPass (%)", fontsize=10)
            ax.set_ylim(0, 100)
            ax.grid(True, alpha=0.35)
    else:
        for group_name, ax in axes_by_group.items():
            ax.set_title(group_name, fontsize=13)
            ax.text(
                0.5,
                0.5,
                "No category\n diagnostics",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.set_ylim(0, 100)
            ax.grid(True, alpha=0.35)

    fig.suptitle(title, fontsize=18, fontweight="bold", y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=220, bbox_inches="tight")

        if save_path.suffix.lower() != ".pdf":
            fig.savefig(save_path.with_suffix(".pdf"), bbox_inches="tight")

    plt.show()
    return fig

In [33]:
plot_paper_ga_dynamics(
    {
        "4B": "/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_allcat_phi35_mini_YYYYMMDD_HHMMSS",
        "7B": "/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_allcat_qwen25_coder_7b_YYYYMMDD_HHMMSS",
        "14B": "/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_allcat_qwen25_coder_14b_YYYYMMDD_HHMMSS",
    },
    save_path="/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/paper_figure1_basic_temporal_complex.png",
    hand_crafted_baseline=25.0,
    cloud_transfer_baseline=50.0,
)

[WARN] no generation progress csv for 4B: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_allcat_phi35_mini_YYYYMMDD_HHMMSS
[WARN] no generation progress csv for 7B: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_allcat_qwen25_coder_7b_YYYYMMDD_HHMMSS
[WARN] no generation progress csv for 14B: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_allcat_qwen25_coder_14b_YYYYMMDD_HHMMSS


FileNotFoundError: No ga_generation_progress.csv or figure1_generation_dynamics.csv found.